# Lesson 07 - အတွက် အစီအစဉ်ဒီဇိုင်းပုံစံ

ဒီ notebook က Microsoft Agent Framework ကို အသုံးပြုတဲ့ AI ကိုယ်စားလှယ်တွေ အတွက် **အစီအစဉ်ဒီဇိုင်းပုံစံ** ကို ပြသပါတယ်။
ခရီးသွားမှု ရှုပ်ထွေးတဲ့ တောင်းဆိုမှုတွေကို ဖွဲ့စည်းထားတဲ့ အစိတ်အပိုင်း ကျွမ်းကျင်သူ ကိုယ်စားလှယ်တွေထံ ခွဲဝေရာ၊
ပြီးတော့ရလာတဲ့ အစီအစဉ်ကို ဖော်ဆောင်ပေးခြင်းကို သင်တွေ့မြင်မှာဖြစ်ပြီး — အားလုံးကို Pydantic မော်ဒယ်တွေနဲ့ အင်အားပေးထားတဲ့ ဖွဲ့စည်းထားတဲ့ အထွက်အနေဖြင့် အသုံးပြုတာပါ။


## တပ်ဆင်ခြင်း


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Task Decomposition

Task decomposition သည် စီမံကိန်းဒီဇိုင်း ပုံစံ၏ အဓိကဖြစ်ပါသည်။ တစ်ဦးတည်းသော အေးဂျင့်ကို ကိစ္စရှုပ်ထွေးသော တောင်းဆိုမှုကို စိတ်တိုင်းမကျအောင် တစ်ဆုံးမှ တစ်ဆုံးကိုင်တွယ်ရန် မမေးဘဲ ပြဿနာကို ပို၍ သေးငယ်ပြီး ရှင်းလင်းသည့် **subtasks** အဖြစ် ခွဲထုတ်သည်။ subtasks တစ်ခုစီကို အထူးပြုအေးဂျင့် (ဥပမာ၊ လေကြောင်း၊ ဟိုတယ်၊ လှုပ်ရှားမှုများ) ‌သတ်မှတ်ပြီး သတ်မှတ်ထားသော ဦးစားပေးမှုများနှင့် အခြေခံမှုအစီအစဉ်ဖြင့် ခန့်ထားသည်။

ဤနည်းလမ်းသည် အချက်အလက် အချို့ကို ပေးသည် –
- **ရှင်းလင်းမှု**: subtasks တစ်ခုစီတွင် တစ်ခုတည်းသောတာဝန်ရှိသည်။
- **병렬 처리**: လွတ်လပ်သော subtasks များကို တပြိုင်နက်တွင် လည်ပတ်နိုင်သည်။
- **ယုံကြည်စိတ်ချရမှု**: မအောင်မြင်မှုများကို subtasks တစ်ခုချင်းစီသို့သာ ကန့်သတ်သည်။
- **ဘတ်ဂျက်ရာဇ၀င်**: ကုန်ကျစရိတ်များကို subtasks သတ်မှတ်ချက်အတိုင်း ခန့်မှန်းပြီး အတည်ပြုသည်။


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## အဆင့်သတ်မှတ်ထားသောထုတ်လွှင့်မှုဖြင့် စီမံကိန်းပြုလုပ်သူအေးဂျင့် တည်ဆောက်ခြင်း

စီမံကိန်းပြုလုပ်သူအေးဂျင့်သည် **ရှေ့ခံဌာန စီစဉ်သူ** အဖြစ် လုပ်ဆောင်သည်။ အဆင့်မြင့် ခရီးသွား တောင်းဆိုမှုကို လက်ခံ၍ အဆိုပါတောင်းဆိုမှုကို ခွဲခြမ်းစိတ်ဖြာ၍ `TravelPlan` အဖြစ် ဖော်ထုတ်ပေးခြင်းဖြစ်သည်။ ၎င်းတွင် အလုပ်ပိုင်းများအား ဦးစားပေးမှုများ သတ်မှတ်ခြင်း၊ နှင့် ဆက်စပ်မှုများကို ဖော်ထုတ်ခြင်းတို့ဖြင့် ကွန်ဆီးယား သို့မဟုတ် ဆောင်ရွက်သည့် အလွှာမှ အလုပ်များကို တာဝန်ယူဆောင်ရွက်နိုင်ရန် ဖြစ်သည်။


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ကွဲပြားသည့်ကိရိယာများဖြင့် အစီအစဉ်တစ်ခုကို အကောင်အထည်ဖော်ခြင်း

အရှေ့တန်းအေးဂျင့်သည် ဖွဲ့စည်းထားသောအစီအစဉ်တစ်ခုကို ထုတ်လုပ်ပြီးနောက်တွင်၊ **ကွန်ဆျာချ်အေးဂျင့်** သည် ၎င်းကို အကောင်အထည်ဖော်သည်။  
မည်သည့်ကိရိယာပညာရှင်သည်တစ်ရှူးအားစီမံခန့်ခွဲရာတွင် (လေယာဉ်ခရီးစဉ်များ၊ ဟိုတယ်များ၊ လှုပ်ရှားမှုများ) တစ်မျိုးစီကို ကိုင်တွယ်သည်။ ကွန်ဆျာချ်သည် အစီအစဉ်၏အောက်ဆုတ်တာငယ်များအားမူတည်သည့် အဆင့်အတိုင်း လည်ပတ်ပြီး သင့်တော်သည့်ကိရိယာတစ်ခုချင်းစီသို့ ပေးပို့သည်။


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ သင်သည် AI အေးဂျင့်များအတွက် **စီမံကိန်းဒီဇိုင်းပုံစံ** ကို သင်ယူခဲ့ပါသည်-

1. **တာဝန်ခွဲခြမ်းခြင်း** — ရုံးအရှေ့ဖက် စီမံကိန်းအေးဂျင့်သည် ရှုပ်ထွေးသောခရီးသွားတောင်းဆိုမှုကို Pydantic မော်ဒယ်များကိုအသုံးပြု၍ ဖွဲ့စည်းထားသော အောက်ပိုင်းတာဝန်များသို့ ခွဲခြားပြီး အထူးပြု အေးဂျင့်တိုင်းအား ဦးစားပေးချက်များနှင့် သက်ဆိုင်မှုများသတ်မှတ်ပေးသည်။
2. **ဖွဲ့စည်းထားသော ထုတ်လွှင့်ချက်** — `response_format` ကို ပေးပို့ခြင်းအားဖြင့် အေးဂျင့်သည် ချေးလွှားစာသားမဟုတ်ဘဲ စစ်ဆေးထားသော `TravelPlan` အရာဝတ္ထုကို ပြန်လည် ထုတ်ပေးပြီး၊ အောက်တန်းလုပ်ဆောင်မှုကို ယုံကြည်စိတ်ချစေသည်။
3. **စီမံကိန်း အကောင်အထည်ဖော်ခြင်း** — ကွန်ဆျာ့ခ် အေးဂျင့်သည် အထူးပြုကိရိယာများဖြစ်သော (`book_flight`, `reserve_hotel`, `book_activity`) ကို အသုံးပြု၍ အောက်ပိုင်းတာဝန်များအား တစ်ကြိမ်ထပ်တလဲလဲ လုပ်ဆောင်ကာ ရလဒ်များကို တင်ပြသည်။

ဒီပုံစံသည် *ဘာလုပ်မည်ကို* (စီမံကိန်း) နှင့် *ဘယ်လိုလုပ်မည်ကို* (အကောင်အထည်ဖော်မှု) ခွဲခြားထားခြင်းဖြစ်ပြီး၊ အေးဂျင့်များအား ပိုမိုအပိုင်းပိုင်းခွဲ၍ စမ်းသပ်နိုင်စေပြီး ချဲ့ထွင်ရလွယ်ကူစေသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အပယ်ကြောင်း**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ မှန်ကန်မှုအတွက် ကြိုးစားပြုလုပ်ထားသော်လည်း၊ မော်တော်မောင်းထားသည့် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း ကျေးဇူးပြု၍ သိရှိထားဆောင်ရွက်ပေးပါရန်။ မူရင်းစာတမ်းကို သူ့ရဲ့ မူလဘာသာဖြင့် အာဏာရှိသော အရင်းခံအရင်းအမြစ်အဖြစ် နှုတ်ဆက်ရပါမည်။ အရေးပါသောသတင်းအချက်အလက်များအတွက် လူ့အတတ်ပညာရှင်တို့၏ ဘာသာပြန်ချက်ကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုရာမှ ဖြစ်ပေါ်လာသော မသိမှတ်မှုများ သို့မဟုတ် မှားယွင်းဖော်ပြချက်များအတွက် ကျွန်ုပ်တို့ ကိစ္စရပ်မကျေမှုရှိပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
